## 01. Setup

In [ ]:
import json
import os
import platform
import time
import warnings
from dataclasses import dataclass
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from joblib import Parallel, delayed
from scipy import stats
from scipy.stats import loguniform, randint, uniform

import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import AdaBoostRegressor, GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import BayesianRidge, Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold, ParameterGrid, RandomizedSearchCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
import statsmodels
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.power import TTestPower
import xgboost
import lightgbm
import catboost
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

SEED = 27
FAST = False
N_REPEATS = 3 if FAST else 10
N_EXPANDED_REPEATS = 1 if FAST else 5
N_ITER_EXPANDED = 10 if FAST else 100
N_BOOT = 2000 if FAST else 10000
N_JOBS = max(1, (os.cpu_count() or 2) - 2)
REPEAT_SEEDS = [SEED + r for r in range(N_REPEATS)]

DATA_DIR = Path("data")
ANALYSIS_PATH = DATA_DIR / "hanwoo_backfat_analysis_dataset.csv"
SCREENING_PATH = DATA_DIR / "hanwoo_backfat_animal_screening.csv"

TARGET = "backfat"
NUMERIC = ["Month", "FatNo", "ColorNo", "Fatcol", "Texture", "Maturity", "agdays", "WBsf"]
FEATURES = NUMERIC + ["Sex"]
SEX_LABELS = {0: "Bull", 1: "Steer", 2: "Cow/heifer"}
SEX_ORDER = ["Bull", "Steer", "Cow/heifer"]

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

INK, INK2, GRID = "#0b0b0b", "#52514e", "#d9d8d4"
BLUE, ORANGE, AQUA, GRAY = "#2a78d6", "#eb6834", "#1baf7a", "#a8a7a2"
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    "axes.unicode_minus": False,
    "font.size": 10,
    "figure.dpi": 110,
})

print(f"Python {platform.python_version()} | scikit-learn {sklearn.__version__} | XGBoost {xgboost.__version__} | "
      f"LightGBM {lightgbm.__version__} | CatBoost {catboost.__version__} | statsmodels {statsmodels.__version__}")
print(f"SEED = {SEED} | FAST = {FAST} | N_REPEATS = {N_REPEATS} | N_JOBS = {N_JOBS}")

## 02. Data

In [ ]:
screening = pd.read_csv(SCREENING_PATH)
data = pd.read_csv(ANALYSIS_PATH)
assert set(data["animal_id"]) == set(screening.loc[screening["included"] == 1, "animal_id"])
data["SexLabel"] = data["Sex"].map(SEX_LABELS)

flow_df = pd.DataFrame([
    ("Source database", len(screening)),
    ("backfat and qngrade recorded", int((screening["exclusion_step"] != 1).sum())),
    ("Month and grading scores recorded", int(screening["included"].sum())),
], columns=["Step", "Animals"])
flow_df["Animals excluded"] = (-flow_df["Animals"].diff()).fillna(0).astype(int)
display(flow_df)

excluded = screening[screening["included"] == 0]
display(pd.crosstab(excluded["source"], excluded["exclusion_reason"], margins=True, margins_name="Total"))
display(pd.crosstab(excluded["exclusion_step"], excluded["sex_label"], margins=True, margins_name="Total"))
display(data.groupby(["Sex", "SexLabel", "sex_label"]).size().rename("n").reset_index())
display(pd.crosstab(data["source"], data["SexLabel"], margins=True, margins_name="Total"))

## 03. Data audit

In [ ]:
INCONSISTENT_IDS = sorted(data.loc[data["record_conflict"] == 1, "animal_id"])
display(data.loc[data["record_conflict"] == 1,
                 ["animal_id", "source", "Month", "Sex", "backfat", "FatNo", "Maturity", "Loinarea", "conflict_note"]])

display(pd.crosstab(data["source"], data["wbsf_cut"], margins=True, margins_name="Total"))
display(pd.crosstab(data["source"], data["agdays"], margins=True, margins_name="Total"))


def eta_squared(values, groups):
    v = pd.Series(np.asarray(values, float))
    g = pd.Series(np.asarray(groups))
    total = ((v - v.mean()) ** 2).sum()
    between = (v.groupby(g).transform("mean") - v.mean()).pow(2).sum()
    return between / total


display(pd.Series({c: eta_squared(data[c], data["source"]) for c in ["WBsf", "agdays", "backfat", "Month", "FatNo"]},
                  name="eta2 (source)").to_frame().T)

## 04. Descriptive statistics

In [ ]:
table1 = data[[TARGET] + NUMERIC].describe().T[["count", "mean", "std", "min", "max"]]
display(table1)

sex_counts = data["SexLabel"].value_counts().reindex(SEX_ORDER).to_frame("n")
sex_counts["%"] = sex_counts["n"] / sex_counts["n"].sum() * 100
display(sex_counts)


def backfat_summary(frame):
    return pd.Series({"n": len(frame), "Mean": frame[TARGET].mean(), "SD": frame[TARGET].std(),
                      "Min": frame[TARGET].min(), "Max": frame[TARGET].max(),
                      "FatNo range": f"{frame['FatNo'].min()}-{frame['FatNo'].max()}"})


s3_table = pd.DataFrame({g: backfat_summary(data[data["SexLabel"] == g]) for g in SEX_ORDER}).T
s3_table.loc["Overall"] = backfat_summary(data)
display(s3_table)

## 05. Correlation analysis

In [ ]:
heatmap_df = pd.get_dummies(data[[TARGET] + FEATURES], columns=["Sex"], prefix="Sex", dtype=int)
heatmap_df = heatmap_df[[TARGET] + NUMERIC + [c for c in heatmap_df.columns if c.startswith("Sex_")]]
cols = heatmap_df.columns
corr_matrix = pd.DataFrame(np.eye(len(cols)), columns=cols, index=cols)
pval_matrix = pd.DataFrame(np.zeros((len(cols), len(cols))), columns=cols, index=cols)
for i in range(len(cols)):
    for j in range(len(cols)):
        if i != j:
            r, p = stats.pearsonr(heatmap_df[cols[i]], heatmap_df[cols[j]])
            corr_matrix.iloc[i, j], pval_matrix.iloc[i, j] = r, p


def significance_stars(p):
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""


annot_matrix = corr_matrix.astype(object).copy()
for i in range(len(cols)):
    for j in range(len(cols)):
        r = corr_matrix.iloc[i, j]
        annot_matrix.iloc[i, j] = f"{r:.2f}" if i == j else f"{r:.2f}{significance_stars(pval_matrix.iloc[i, j])}"

plt.figure(figsize=(13, 11))
sns.heatmap(corr_matrix.astype(float), annot=annot_matrix, fmt="", cmap="coolwarm", vmin=-1, vmax=1, center=0,
            square=True, cbar=True, linewidths=0.5, linecolor="white", annot_kws={"size": 8})
plt.title("Correlation Heatmap with Significance Levels (*: p<0.05, **: p<0.01, ***: p<0.001)")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

display(pd.DataFrame({"r": corr_matrix[TARGET], "p": pval_matrix[TARGET]}).drop(TARGET).sort_values("r", key=np.abs, ascending=False))

## 06. Preprocessing and encoding check

In [ ]:
def make_pipeline(model, numeric, categorical, drop=None):
    transformers = []
    if numeric:
        transformers.append(("num", StandardScaler(), list(numeric)))
    if categorical:
        transformers.append(("sex", OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop=drop), list(categorical)))
    pre = ColumnTransformer(transformers, remainder="drop", verbose_feature_names_out=False)
    return Pipeline([("preprocess", pre), ("model", model)])


X_all, y_all = data[FEATURES], data[TARGET]
fold_check = []
for k, (tr, te) in enumerate(KFold(n_splits=5, shuffle=True, random_state=SEED).split(X_all), start=1):
    pipe = make_pipeline(LinearRegression(), NUMERIC, ["Sex"]).fit(X_all.iloc[tr], y_all.iloc[tr])
    pre = pipe.named_steps["preprocess"]
    z_test = pd.DataFrame(pre.transform(X_all.iloc[te]), columns=pre.get_feature_names_out())
    scaler = pre.named_transformers_["num"]
    fold_check.append({
        "Fold": k, "n train": len(tr), "n test": len(te),
        "scaler mean Month": scaler.mean_[0],
        "scaler mean WBsf": scaler.mean_[NUMERIC.index("WBsf")],
        "test mean z(Month)": z_test["Month"].mean(),
        "Sex indicator row sums": z_test[["Sex_0", "Sex_1", "Sex_2"]].sum(axis=1).unique().tolist(),
    })
print("Design matrix:", list(pre.get_feature_names_out()))
display(pd.DataFrame(fold_check))

In [ ]:
def oof_predict(factory, X, y, seed=SEED):
    pred = np.empty(len(y))
    for tr, te in KFold(n_splits=5, shuffle=True, random_state=seed).split(X):
        pred[te] = factory().fit(X.iloc[tr], y.iloc[tr]).predict(X.iloc[te])
    return pred


p_all3 = oof_predict(lambda: make_pipeline(LinearRegression(), NUMERIC, ["Sex"]), X_all, y_all)
p_ref = oof_predict(lambda: make_pipeline(LinearRegression(), NUMERIC, ["Sex"], drop="first"), X_all, y_all)
p_noint = oof_predict(lambda: make_pipeline(LinearRegression(fit_intercept=False), NUMERIC, ["Sex"]), X_all, y_all)
print(f"OLS OOF max |diff|, 3 indicators vs bull reference: {np.max(np.abs(p_all3 - p_ref)):.2e}")
print(f"OLS OOF max |diff|, fit_intercept True vs False:     {np.max(np.abs(p_all3 - p_noint)):.2e}")

sex_dummies = pd.get_dummies(data["Sex"].map(SEX_LABELS), dtype=float)[["Steer", "Cow/heifer"]]
X_vif = sm.add_constant(pd.concat([data[NUMERIC].astype(float), sex_dummies], axis=1))
vif = pd.Series([variance_inflation_factor(X_vif.values, i) for i in range(1, X_vif.shape[1])],
                index=X_vif.columns[1:], name="VIF")
display(vif.round(2).to_frame().T)

## 07. Models and nested cross-validation

In [ ]:
@dataclass
class ModelSpec:
    name: str
    estimator: object
    param_grid: dict


MODEL_SPECS = [
    ModelSpec("LinearRegression", LinearRegression(), {"model__fit_intercept": [True, False]}),
    ModelSpec("Ridge", Ridge(random_state=SEED), {"model__alpha": [0.1, 1.0, 10.0, 100.0]}),
    ModelSpec("Lasso", Lasso(max_iter=10000, random_state=SEED), {"model__alpha": [0.001, 0.01, 0.1, 1.0]}),
    ModelSpec("BayesianRidge", BayesianRidge(), {"model__alpha_1": [1e-6, 1e-5], "model__lambda_1": [1e-6, 1e-5]}),
    ModelSpec("RandomForest", RandomForestRegressor(random_state=SEED, n_jobs=1),
              {"model__n_estimators": [100, 200], "model__max_depth": [5, 10, None], "model__min_samples_leaf": [1, 3]}),
    ModelSpec("GradientBoosting", GradientBoostingRegressor(random_state=SEED),
              {"model__n_estimators": [100, 200], "model__learning_rate": [0.03, 0.1], "model__max_depth": [2, 3]}),
    ModelSpec("AdaBoost", AdaBoostRegressor(random_state=SEED),
              {"model__n_estimators": [50, 100, 200], "model__learning_rate": [0.03, 0.1, 0.3]}),
    ModelSpec("KNN", KNeighborsRegressor(), {"model__n_neighbors": [3, 5, 7, 9, 11], "model__weights": ["uniform", "distance"]}),
    ModelSpec("SVR", SVR(), {"model__kernel": ["rbf", "sigmoid"], "model__C": [1, 10, 100, 1000],
                             "model__gamma": ["scale", 0.001, 0.01, 0.1], "model__epsilon": [0.1, 0.2]}),
    ModelSpec("DecisionTree", DecisionTreeRegressor(random_state=SEED),
              {"model__max_depth": [3, 5, 10, None], "model__min_samples_leaf": [1, 3, 5]}),
    ModelSpec("MLP", MLPRegressor(max_iter=1500, early_stopping=True, random_state=SEED),
              {"model__hidden_layer_sizes": [(50,), (100,), (50, 25)], "model__alpha": [1e-4, 1e-3]}),
    ModelSpec("XGBoost", XGBRegressor(objective="reg:squarederror", random_state=SEED, n_jobs=1, tree_method="hist", verbosity=0),
              {"model__n_estimators": [100, 200], "model__learning_rate": [0.03, 0.1], "model__max_depth": [2, 3],
               "model__subsample": [0.8, 1.0]}),
    ModelSpec("LightGBM", LGBMRegressor(random_state=SEED, n_jobs=1, verbosity=-1),
              {"model__n_estimators": [100, 200], "model__learning_rate": [0.03, 0.1], "model__num_leaves": [15, 31],
               "model__min_child_samples": [5, 10]}),
    ModelSpec("CatBoost", CatBoostRegressor(random_seed=SEED, verbose=0, allow_writing_files=False, thread_count=1),
              {"model__iterations": [100, 200], "model__learning_rate": [0.03, 0.1], "model__depth": [3, 5]}),
]
DUMMY_SPEC = ModelSpec("Dummy", DummyRegressor(), {"model__strategy": ["mean", "median"]})
SPEC_BY_NAME = {s.name: s for s in MODEL_SPECS}

s1_table = pd.DataFrame({
    "Model": [s.name for s in MODEL_SPECS],
    "Grid": ["; ".join(f"{k.replace('model__', '')} = {v}" for k, v in s.param_grid.items()) for s in MODEL_SPECS],
    "Configurations": [len(ParameterGrid(s.param_grid)) for s in MODEL_SPECS],
})
display(s1_table)
print("Total configurations:", s1_table["Configurations"].sum())

In [ ]:
def _fit_outer_fold(spec, X, y, train_idx, test_idx, numeric, categorical, seed, search, n_iter):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        inner = KFold(n_splits=3, shuffle=True, random_state=seed)
        pipe = make_pipeline(clone(spec.estimator), numeric, categorical)
        common = dict(scoring="neg_mean_absolute_error", cv=inner, n_jobs=1, refit=True, error_score="raise")
        if search == "grid":
            tuner = GridSearchCV(pipe, spec.param_grid, **common)
        else:
            tuner = RandomizedSearchCV(pipe, spec.param_grid, n_iter=n_iter, random_state=seed, **common)
        tuner.fit(X.iloc[train_idx], y.iloc[train_idx])
        pred = tuner.predict(X.iloc[test_idx])
    y_test = y.iloc[test_idx].to_numpy()
    metrics = {
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": mean_squared_error(y_test, pred) ** 0.5,
        "R2": r2_score(y_test, pred),
        "InnerCV_MAE": -tuner.best_score_,
        "BestParams": json.dumps(tuner.best_params_, sort_keys=True, default=str),
    }
    return metrics, pred


def run_nested_cv(frame, features, specs, seeds, scenario, search="grid", n_iter=None):
    numeric = [f for f in features if f != "Sex"]
    categorical = ["Sex"] if "Sex" in features else []
    X, y = frame[features], frame[TARGET]
    tasks = []
    for r, seed in enumerate(seeds):
        outer = KFold(n_splits=5, shuffle=True, random_state=seed)
        for k, (tr, te) in enumerate(outer.split(X), start=1):
            for spec in specs:
                tasks.append((r, seed, k, spec, tr, te))
    t0 = time.time()
    results = Parallel(n_jobs=N_JOBS)(
        delayed(_fit_outer_fold)(spec, X, y, tr, te, numeric, categorical, seed, search, n_iter)
        for (r, seed, k, spec, tr, te) in tasks
    )
    fold_rows, oof_parts = [], []
    for (r, seed, k, spec, tr, te), (m, pred) in zip(tasks, results):
        fold_rows.append({"Scenario": scenario, "Repeat": r, "Seed": seed, "Fold": k, "Model": spec.name, **m})
        oof_parts.append(pd.DataFrame({"Scenario": scenario, "Repeat": r, "Fold": k, "Model": spec.name,
                                       "row": frame.index[te], "y": y.iloc[te].to_numpy(), "pred": pred}))
    print(f"[{scenario}] n = {len(frame)}, {len(specs)} models x {len(seeds)} repeats x 5 folds, {time.time() - t0:.0f} s")
    return pd.DataFrame(fold_rows), pd.concat(oof_parts, ignore_index=True)


def seed_table(fold_df, repeat=0):
    d = fold_df[fold_df["Repeat"] == repeat]
    return (d.groupby("Model").agg(MAE_mean=("MAE", "mean"), MAE_sd=("MAE", "std"), RMSE_mean=("RMSE", "mean"),
                                   RMSE_sd=("RMSE", "std"), R2_mean=("R2", "mean"), R2_sd=("R2", "std"))
             .sort_values("MAE_mean"))


def per_repeat(fold_df):
    rep = fold_df.groupby(["Model", "Repeat"], as_index=False)[["MAE", "RMSE", "R2"]].mean()
    rep["rank_MAE"] = rep.groupby("Repeat")["MAE"].rank(method="min")
    rep["rank_R2"] = rep.groupby("Repeat")["R2"].rank(method="min", ascending=False)
    return rep


def repeat_table(fold_df):
    rep = per_repeat(fold_df)
    t = rep.groupby("Model").agg(
        MAE=("MAE", "mean"), MAE_sd_rep=("MAE", "std"), RMSE=("RMSE", "mean"),
        R2=("R2", "mean"), R2_sd_rep=("R2", "std"),
        rank_MAE=("rank_MAE", "mean"), rank_R2=("rank_R2", "mean"),
        first_MAE=("rank_MAE", lambda s: int((s == 1).sum())),
        first_R2=("rank_R2", lambda s: int((s == 1).sum())),
    )
    t["MAE_sd_fold"] = fold_df.groupby("Model")["MAE"].std()
    t["R2_sd_fold"] = fold_df.groupby("Model")["R2"].std()
    return t.sort_values("MAE")


def paired_fold_diff(fold_df, a, b, metric):
    p = fold_df[fold_df["Model"].isin([a, b])].pivot_table(index=["Repeat", "Fold"], columns="Model", values=metric)
    return (p[a] - p[b]).to_numpy()


def corrected_ttest(diffs, n_folds=5):
    d = np.asarray(diffs, float)
    J = d.size
    se = np.sqrt((1.0 / J + 1.0 / (n_folds - 1)) * d.var(ddof=1))
    t = d.mean() / se
    return d.mean(), t, 2 * stats.t.sf(abs(t), df=J - 1)


def kendalls_w(rank_matrix):
    R = np.asarray(rank_matrix, float)
    m, n = R.shape
    S = ((R.sum(axis=0) - R.sum(axis=0).mean()) ** 2).sum()
    return 12 * S / (m ** 2 * (n ** 3 - n))


def group_metrics(oof_df, frame, group_col):
    d = oof_df.join(frame[group_col], on="row")
    out = (d.groupby(["Model", "Repeat", group_col])[["y", "pred"]]
             .apply(lambda t: pd.Series({"n": len(t), "MAE": mean_absolute_error(t["y"], t["pred"]),
                                         "R2": r2_score(t["y"], t["pred"])})))
    return out.groupby(["Model", group_col]).mean()


def pooled_metrics(oof_df):
    out = (oof_df.groupby(["Model", "Repeat"])[["y", "pred"]]
             .apply(lambda t: pd.Series({"MAE": mean_absolute_error(t["y"], t["pred"]),
                                         "RMSE": mean_squared_error(t["y"], t["pred"]) ** 0.5,
                                         "R2": r2_score(t["y"], t["pred"])})))
    return out.groupby("Model").mean()

## 08. Benchmark under nested cross-validation

In [ ]:
fold_main, oof_main = run_nested_cv(data, FEATURES, MODEL_SPECS, REPEAT_SEEDS, "All")

table2 = seed_table(fold_main, repeat=0)
display(table2)

s2_table = (fold_main[fold_main["Repeat"] == 0]
            .set_index(["Model", "Fold"])[["MAE", "RMSE", "R2"]]
            .loc[table2.index])
display(s2_table)
display(pooled_metrics(oof_main[oof_main["Repeat"] == 0]).loc[table2.index])

In [ ]:
ordered_perf = table2.reset_index()
colors = ["#d94f3d" if i == 0 else "#5b9bd5" for i in range(len(ordered_perf))]
fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.bar(ordered_perf["Model"], ordered_perf["MAE_mean"], yerr=ordered_perf["MAE_sd"], color=colors,
              edgecolor="black", linewidth=0.6, capsize=3)
ax.set_ylabel("MAE")
ax.set_title("Nested K-fold CV MAE by Model")
ax.set_ylim(0, ordered_perf["MAE_mean"].max() + ordered_perf["MAE_sd"].max() + 0.6)
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.set_axisbelow(True)
plt.xticks(rotation=45, ha="right")
for bar, value in zip(bars, ordered_perf["MAE_mean"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05, f"{value:.2f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

ordered_r2 = table2.sort_values("R2_mean", ascending=False).reset_index()
colors = ["#d94f3d" if i == 0 else "#5b9bd5" for i in range(len(ordered_r2))]
fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.bar(ordered_r2["Model"], ordered_r2["R2_mean"], yerr=ordered_r2["R2_sd"], color=colors,
              edgecolor="black", linewidth=0.6, capsize=3)
ax.set_ylabel("R2")
ax.set_title("Nested K-fold CV R2 by Model")
ax.axhline(0, color="black", linewidth=0.8)
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.set_axisbelow(True)
plt.xticks(rotation=45, ha="right")
for bar, value in zip(bars, ordered_r2["R2_mean"]):
    ax.text(bar.get_x() + bar.get_width() / 2, value + 0.015 if value >= 0 else value - 0.04, f"{value:.3f}",
            ha="center", va="bottom" if value >= 0 else "top", fontsize=9)
plt.tight_layout()
plt.show()

## 09. Sex-stratified prediction profiles

In [ ]:
profile_models = ["Lasso", "SVR"]
prof = (oof_main[(oof_main["Repeat"] == 0) & oof_main["Model"].isin(profile_models)]
        .join(data[["Sex", "FatNo"]], on="row"))
styles = {"Lasso": dict(label="Mean Predicted (Lasso)", color="#ff7f0e", linestyle="-", linewidth=2.6),
          "SVR": dict(label="Mean Predicted (SVR)", color="#2ca02c", linestyle="-.", linewidth=2.4)}

fig, axes = plt.subplots(1, 3, figsize=(18.3, 4.8), sharey=True)
for ax, sex in zip(axes, [0, 1, 2]):
    actual = prof[(prof["Model"] == "Lasso") & (prof["Sex"] == sex)]
    actual_mean = actual.groupby("FatNo", as_index=False)["y"].mean()
    ax.scatter(actual["FatNo"], actual["y"], label="Actual", alpha=0.24, s=32, color="#5b9bd5", edgecolor="none")
    ax.plot(actual_mean["FatNo"], actual_mean["y"], label="Mean Actual", linestyle="--", linewidth=2.0, color="#1f77b4")
    for m in profile_models:
        pm = prof[(prof["Model"] == m) & (prof["Sex"] == sex)].groupby("FatNo", as_index=False)["pred"].mean()
        ax.plot(pm["FatNo"], pm["pred"], **styles[m])
    ax.set_title(SEX_LABELS[sex])
    ax.set_xlabel("Fat Number")
    ax.grid(True, linestyle="--", linewidth=0.6, alpha=0.45)
    ax.set_axisbelow(True)
axes[0].set_ylabel("Backfat")
handles, labels = axes[-1].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.98), ncol=4, frameon=True)
fig.subplots_adjust(top=0.76, bottom=0.16, wspace=0.08)
plt.show()

oof0 = oof_main[(oof_main["Repeat"] == 0) & oof_main["Model"].isin(profile_models)].join(data["SexLabel"], on="row")
s4_rows = []
for m in profile_models:
    for g in SEX_ORDER + ["All"]:
        d = oof0[oof0["Model"] == m] if g == "All" else oof0[(oof0["Model"] == m) & (oof0["SexLabel"] == g)]
        s4_rows.append({"Model": m, "Sex": g, "n": len(d), "MAE": mean_absolute_error(d["y"], d["pred"]),
                        "R2": r2_score(d["y"], d["pred"])})
display(pd.DataFrame(s4_rows))

## 10. SVR vs Lasso

In [ ]:
f0 = fold_main[fold_main["Repeat"] == 0].pivot(index="Fold", columns="Model", values="MAE")[["SVR", "Lasso"]]
f0["SVR - Lasso"] = f0["SVR"] - f0["Lasso"]
display(f0)
t_res = stats.ttest_rel(f0["SVR"], f0["Lasso"])
w_res = stats.wilcoxon(f0["SVR"], f0["Lasso"])
w_min = stats.wilcoxon(np.arange(1, 6)).pvalue
sd_diff = f0["SVR - Lasso"].std(ddof=1)
d_detect = TTestPower().solve_power(nobs=5, alpha=0.05, power=0.8, alternative="two-sided")
print(f"paired t = {t_res.statistic:.2f}, p = {t_res.pvalue:.3f} | Wilcoxon W = {w_res.statistic:.0f}, p = {w_res.pvalue:.3f}")
print(f"minimum attainable Wilcoxon p (n = 5) = {w_min:.4f}")
print(f"mean difference = {f0['SVR - Lasso'].mean():+.3f} mm, SD = {sd_diff:.3f} mm, "
      f"detectable difference (80% power) = {d_detect * sd_diff:.3f} mm")

In [ ]:
o0 = oof_main[oof_main["Repeat"] == 0].pivot_table(index="row", columns="Model", values="pred")
yv = data.loc[o0.index, TARGET].to_numpy().astype(float)
p_svr, p_las = o0["SVR"].to_numpy(), o0["Lasso"].to_numpy()

rng = np.random.default_rng(SEED)
idx = rng.integers(0, len(yv), size=(N_BOOT, len(yv)))
yb, sb, lb = yv[idx], p_svr[idx], p_las[idx]
sst = ((yb - yb.mean(axis=1, keepdims=True)) ** 2).sum(axis=1)
boot = pd.DataFrame({
    "MAE (SVR - Lasso)": np.abs(yb - sb).mean(axis=1) - np.abs(yb - lb).mean(axis=1),
    "RMSE (SVR - Lasso)": np.sqrt(((yb - sb) ** 2).mean(axis=1)) - np.sqrt(((yb - lb) ** 2).mean(axis=1)),
    "R2 (SVR - Lasso)": (1 - ((yb - sb) ** 2).sum(axis=1) / sst) - (1 - ((yb - lb) ** 2).sum(axis=1) / sst),
})
point = {
    "MAE (SVR - Lasso)": mean_absolute_error(yv, p_svr) - mean_absolute_error(yv, p_las),
    "RMSE (SVR - Lasso)": mean_squared_error(yv, p_svr) ** 0.5 - mean_squared_error(yv, p_las) ** 0.5,
    "R2 (SVR - Lasso)": r2_score(yv, p_svr) - r2_score(yv, p_las),
}
boot_ci = pd.DataFrame({"Difference": point, "CI 2.5%": boot.quantile(0.025), "CI 97.5%": boot.quantile(0.975),
                        "P(diff < 0)": (boot < 0).mean()})
display(boot_ci)
print(f"Wilcoxon on paired absolute errors (n = {len(yv)}): p = {stats.wilcoxon(np.abs(yv - p_svr), np.abs(yv - p_las)).pvalue:.3f}")

In [ ]:
rep_main = per_repeat(fold_main)
rep_sl = rep_main[rep_main["Model"].isin(["SVR", "Lasso"])].pivot(index="Repeat", columns="Model", values=["MAE", "R2"])
rep_sl.columns = [f"{m} {mod}" for m, mod in rep_sl.columns]
rep_sl.index = [f"seed {s}" for s in REPEAT_SEEDS]
display(rep_sl)
print(f"SVR lower MAE: {(rep_sl['MAE SVR'] < rep_sl['MAE Lasso']).sum()}/{N_REPEATS}; "
      f"Lasso higher R2: {(rep_sl['R2 Lasso'] > rep_sl['R2 SVR']).sum()}/{N_REPEATS}")

rows = []
for metric in ["MAE", "RMSE", "R2"]:
    mean_d, t_c, p_c = corrected_ttest(paired_fold_diff(fold_main, "SVR", "Lasso", metric))
    rows.append({"Metric": metric, "SVR - Lasso": mean_d, "t": t_c, "p (corrected)": p_c})
display(pd.DataFrame(rows).set_index("Metric"))

In [ ]:
rows = []
for spec in MODEL_SPECS:
    if spec.name == "Lasso":
        continue
    rec = {"Model": spec.name}
    for metric in ["MAE", "R2"]:
        mean_d, _, p_c = corrected_ttest(paired_fold_diff(fold_main, spec.name, "Lasso", metric))
        rec[f"d{metric}"] = mean_d
        rec[f"p {metric}"] = p_c
    rows.append(rec)
vs_lasso = pd.DataFrame(rows).set_index("Model")
for metric in ["MAE", "R2"]:
    vs_lasso[f"p {metric} (Holm)"] = multipletests(vs_lasso[f"p {metric}"], method="holm")[1]
display(vs_lasso[["dMAE", "p MAE", "p MAE (Holm)", "dR2", "p R2", "p R2 (Holm)"]].sort_values("dR2"))

## 11. Ranking stability across partitions

In [ ]:
stab = repeat_table(fold_main)
display(stab)
w_mae = kendalls_w(rep_main.pivot(index="Repeat", columns="Model", values="rank_MAE"))
w_r2 = kendalls_w(rep_main.pivot(index="Repeat", columns="Model", values="rank_R2"))
rho_seed = stats.spearmanr(table2.loc[stab.index, "MAE_mean"], stab["MAE"]).statistic
print(f"Kendall's W: MAE {w_mae:.2f}, R2 {w_r2:.2f} | Spearman rho (seed {SEED} vs mean): {rho_seed:.2f}")

order = list(stab.index)
highlight = {"SVR": BLUE, "Lasso": ORANGE}
fig, axes = plt.subplots(1, 2, figsize=(11, 5.4), sharey=True)
jit = np.random.default_rng(0).uniform(-0.18, 0.18, size=N_REPEATS)
for ax, metric, xlabel in zip(axes, ["MAE", "R2"], ["MAE (mm)", "R²"]):
    for i, m in enumerate(order):
        v = rep_main.loc[rep_main["Model"] == m, metric].to_numpy()
        ax.scatter(v, i + jit[: v.size], s=16, color=highlight.get(m, GRAY), alpha=0.85, linewidths=0, zorder=3)
        ax.plot([v.mean()] * 2, [i - 0.32, i + 0.32], color=INK, lw=2, zorder=4)
    ax.set_xlabel(xlabel)
    ax.grid(axis="x", lw=0.6, color=GRID)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_yticks(range(len(order)))
axes[0].set_yticklabels(order)
axes[0].invert_yaxis()
handles = [plt.Line2D([], [], marker="o", ls="", color=c, label=l) for l, c in [("SVR", BLUE), ("Lasso", ORANGE), ("Other", GRAY)]]
handles.append(plt.Line2D([], [], color=INK, lw=2, label="Mean"))
fig.legend(handles=handles, loc="upper center", ncol=4, frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.93))
plt.show()

## 12. Sex subgroup sensitivity

In [ ]:
gm = group_metrics(oof_main, data, "SexLabel")
within = gm[["MAE", "R2"]].unstack("SexLabel")
within.columns = [f"{m} {g}" for m, g in within.columns]
within = within[[f"{m} {g}" for m in ["MAE", "R2"] for g in SEX_ORDER]]
within = within.join(pooled_metrics(oof_main)[["MAE", "R2"]].add_suffix(" pooled")).loc[stab.index]
display(within)

In [ ]:
data_nobull = data[data["Sex"] != 0]
data_steer = data[data["Sex"] == 1]
fold_nobull, oof_nobull = run_nested_cv(data_nobull, FEATURES, MODEL_SPECS, REPEAT_SEEDS, "No bulls")
fold_steer, oof_steer = run_nested_cv(data_steer, NUMERIC, MODEL_SPECS, REPEAT_SEEDS, "Steers only")

tab_all, tab_nb, tab_st = repeat_table(fold_main), repeat_table(fold_nobull), repeat_table(fold_steer)
compare = pd.concat({f"All ({len(data)})": tab_all[["MAE", "R2", "rank_MAE", "rank_R2"]],
                     f"No bulls ({len(data_nobull)})": tab_nb[["MAE", "R2", "rank_MAE", "rank_R2"]],
                     f"Steers ({len(data_steer)})": tab_st[["MAE", "R2", "rank_MAE", "rank_R2"]]}, axis=1).loc[order]
display(compare)

rows = []
for label, tab in [("No bulls", tab_nb), ("Steers only", tab_st)]:
    for metric in ["MAE", "R2"]:
        a, b = tab_all.loc[order, metric], tab.loc[order, metric]
        best = (lambda s: s.idxmin()) if metric == "MAE" else (lambda s: s.idxmax())
        top4 = (lambda s: s.nsmallest(4)) if metric == "MAE" else (lambda s: s.nlargest(4))
        rows.append({"Scenario": label, "Metric": metric,
                     "Spearman rho": stats.spearmanr(a, b).statistic, "Kendall tau": stats.kendalltau(a, b).statistic,
                     "Best (All)": best(a), "Best (scenario)": best(b),
                     "Top-4 overlap": len(set(top4(a).index) & set(top4(b).index))})
display(pd.DataFrame(rows))

for label, fd in [("No bulls", fold_nobull), ("Steers only", fold_steer)]:
    out = []
    for metric in ["MAE", "R2"]:
        mean_d, _, p_c = corrected_ttest(paired_fold_diff(fd, "SVR", "Lasso", metric))
        out.append(f"{metric} SVR - Lasso = {mean_d:+.3f} (p = {p_c:.3f})")
    print(f"{label}: " + " | ".join(out))

In [ ]:
def mae_r2_by_model(o):
    return (o.groupby(["Model", "Repeat"])[["y", "pred"]]
              .apply(lambda t: pd.Series({"MAE": mean_absolute_error(t["y"], t["pred"]), "R2": r2_score(t["y"], t["pred"])}))
              .groupby("Model").mean())


with_bulls = oof_main[oof_main["row"].isin(data_nobull.index)]
matched = pd.concat({"Trained with bulls": mae_r2_by_model(with_bulls),
                     "Trained without bulls": mae_r2_by_model(oof_nobull)}, axis=1)
matched[("Difference", "MAE")] = matched[("Trained without bulls", "MAE")] - matched[("Trained with bulls", "MAE")]
matched[("Difference", "R2")] = matched[("Trained without bulls", "R2")] - matched[("Trained with bulls", "R2")]
display(matched.loc[order])

fig, axes = plt.subplots(1, 2, figsize=(11, 5.4), sharey=True)
scen = [(f"All (n={len(data)})", tab_all, BLUE, "o", -0.2), (f"No bulls (n={len(data_nobull)})", tab_nb, ORANGE, "s", 0.0),
        (f"Steers only (n={len(data_steer)})", tab_st, AQUA, "D", 0.2)]
for ax, metric, xlabel in zip(axes, ["MAE", "R2"], ["MAE (mm)", "R²"]):
    for name, tab, color, marker, off in scen:
        ax.scatter(tab.loc[order, metric], np.arange(len(order)) + off, color=color, marker=marker, s=34,
                   label=name, zorder=3, edgecolors="white", linewidths=0.8)
    ax.set_xlabel(f"{xlabel}, mean of {N_REPEATS} repeats")
    ax.grid(axis="x", lw=0.6, color=GRID)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_yticks(range(len(order)))
axes[0].set_yticklabels(order)
axes[0].invert_yaxis()
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.93))
plt.show()

## 13. Expanded hyperparameter search

In [ ]:
EXPANDED_SPECS = [
    ModelSpec("RandomForest", RandomForestRegressor(random_state=SEED, n_jobs=1), {
        "model__n_estimators": randint(100, 1001),
        "model__max_depth": [2, 3, 4, 5, 6, 8, 10, 15, None],
        "model__min_samples_leaf": randint(1, 31),
        "model__max_features": [1.0, 0.8, 0.6, 0.4, "sqrt"],
        "model__max_samples": [None, 0.5, 0.7, 0.9],
    }),
    ModelSpec("GradientBoosting", GradientBoostingRegressor(random_state=SEED), {
        "model__n_estimators": randint(50, 1001),
        "model__learning_rate": loguniform(0.005, 0.3),
        "model__max_depth": randint(1, 6),
        "model__min_samples_leaf": randint(1, 31),
        "model__subsample": uniform(0.5, 0.5),
        "model__max_features": [None, 0.8, 0.6, "sqrt"],
        "model__loss": ["squared_error", "absolute_error", "huber"],
    }),
    ModelSpec("AdaBoost", AdaBoostRegressor(estimator=DecisionTreeRegressor(max_depth=3, random_state=SEED), random_state=SEED), {
        "model__n_estimators": randint(25, 501),
        "model__learning_rate": loguniform(0.005, 1.0),
        "model__loss": ["linear", "square", "exponential"],
        "model__estimator__max_depth": [1, 2, 3, 4, 5, 6],
        "model__estimator__min_samples_leaf": [1, 3, 5, 10, 20],
    }),
    ModelSpec("XGBoost", XGBRegressor(objective="reg:squarederror", random_state=SEED, n_jobs=1, tree_method="hist", verbosity=0), {
        "model__n_estimators": randint(50, 1001),
        "model__learning_rate": loguniform(0.005, 0.3),
        "model__max_depth": randint(1, 7),
        "model__min_child_weight": loguniform(1, 30),
        "model__subsample": uniform(0.5, 0.5),
        "model__colsample_bytree": uniform(0.5, 0.5),
        "model__reg_alpha": loguniform(1e-4, 10),
        "model__reg_lambda": loguniform(1e-3, 100),
    }),
    ModelSpec("LightGBM", LGBMRegressor(random_state=SEED, n_jobs=1, verbosity=-1, subsample_freq=1), {
        "model__n_estimators": randint(50, 1001),
        "model__learning_rate": loguniform(0.005, 0.3),
        "model__num_leaves": randint(2, 64),
        "model__max_depth": [-1, 2, 3, 4, 6],
        "model__min_child_samples": randint(5, 61),
        "model__subsample": uniform(0.5, 0.5),
        "model__colsample_bytree": uniform(0.5, 0.5),
        "model__reg_alpha": loguniform(1e-4, 10),
        "model__reg_lambda": loguniform(1e-3, 100),
    }),
    ModelSpec("CatBoost", CatBoostRegressor(random_seed=SEED, verbose=0, allow_writing_files=False, thread_count=1), {
        "model__iterations": randint(100, 1001),
        "model__learning_rate": loguniform(0.01, 0.3),
        "model__depth": randint(2, 9),
        "model__l2_leaf_reg": loguniform(1, 30),
        "model__random_strength": loguniform(0.1, 10),
    }),
    ModelSpec("MLP", MLPRegressor(max_iter=2000, early_stopping=True, random_state=SEED), {
        "model__hidden_layer_sizes": [(8,), (16,), (32,), (64,), (128,), (16, 8), (32, 16), (64, 32), (128, 64), (64, 32, 16)],
        "model__activation": ["relu", "tanh"],
        "model__alpha": loguniform(1e-5, 10),
        "model__learning_rate_init": loguniform(1e-4, 1e-2),
        "model__batch_size": [16, 32, 64, "auto"],
    }),
]
EXP_SEEDS = REPEAT_SEEDS[:N_EXPANDED_REPEATS]
fold_exp, oof_exp = run_nested_cv(data, FEATURES, EXPANDED_SPECS, EXP_SEEDS, "Expanded search",
                                  search="random", n_iter=N_ITER_EXPANDED)

In [ ]:
grid_same = fold_main[fold_main["Repeat"] < N_EXPANDED_REPEATS]
tab_grid, tab_exp = repeat_table(grid_same), repeat_table(fold_exp)
exp_names = [s.name for s in EXPANDED_SPECS]
lasso_ref = tab_grid.loc["Lasso"]
print(f"Reference (same partitions): Lasso MAE {lasso_ref['MAE']:.3f}, R2 {lasso_ref['R2']:.3f} | "
      f"SVR MAE {tab_grid.loc['SVR', 'MAE']:.3f}, R2 {tab_grid.loc['SVR', 'R2']:.3f}")

rows = []
for m in exp_names:
    both = pd.concat([fold_exp[fold_exp["Model"] == m], grid_same[grid_same["Model"] == "Lasso"]])
    _, _, p_mae = corrected_ttest(paired_fold_diff(both, m, "Lasso", "MAE"))
    _, _, p_r2 = corrected_ttest(paired_fold_diff(both, m, "Lasso", "R2"))
    rows.append({
        "Model": m,
        "Configurations": f"{len(ParameterGrid(SPEC_BY_NAME[m].param_grid))} -> {N_ITER_EXPANDED}",
        "MAE grid": tab_grid.loc[m, "MAE"], "MAE expanded": tab_exp.loc[m, "MAE"],
        "dMAE": tab_exp.loc[m, "MAE"] - tab_grid.loc[m, "MAE"],
        "R2 grid": tab_grid.loc[m, "R2"], "R2 expanded": tab_exp.loc[m, "R2"],
        "dR2": tab_exp.loc[m, "R2"] - tab_grid.loc[m, "R2"],
        "R2 - Lasso": tab_exp.loc[m, "R2"] - lasso_ref["R2"],
        "p vs Lasso MAE": p_mae, "p vs Lasso R2": p_r2,
        "Inner-CV MAE": fold_exp.loc[fold_exp["Model"] == m, "InnerCV_MAE"].mean(),
    })
exp_table = pd.DataFrame(rows).set_index("Model").sort_values("R2 expanded", ascending=False)
display(exp_table)


def summarize_params(fd, model):
    p = pd.DataFrame([json.loads(s) for s in fd.loc[fd["Model"] == model, "BestParams"]])
    out = {}
    for c in p.columns:
        as_text = p[c].astype(str)
        try:
            num = pd.to_numeric(p[c], errors="coerce")
        except TypeError:
            num = pd.Series(np.nan, index=p.index)
        if num.notna().all() and as_text.nunique() > 3:
            out[c.replace("model__", "")] = f"median {num.median():.3g}"
        else:
            out[c.replace("model__", "")] = as_text.value_counts().idxmax()
    return out


display(pd.DataFrame({m: summarize_params(fold_exp, m) for m in exp_names}).T)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), sharey=True)
ypos = np.arange(len(exp_table))
for ax, metric, xlabel in zip(axes, ["MAE", "R2"], ["MAE (mm)", "R²"]):
    g, e = exp_table[f"{metric} grid"], exp_table[f"{metric} expanded"]
    ax.hlines(ypos, g, e, color=GRID, lw=2.5, zorder=1)
    ax.scatter(g, ypos, color=GRAY, s=40, zorder=3, label="Grid", edgecolors="white", linewidths=0.8)
    ax.scatter(e, ypos, color=BLUE, s=40, zorder=3, label=f"Randomized search ({N_ITER_EXPANDED})", edgecolors="white", linewidths=0.8)
    ax.axvline(lasso_ref[metric], color=ORANGE, ls="--", lw=1.5, label="Lasso")
    ax.set_xlabel(f"{xlabel}, mean of {N_EXPANDED_REPEATS} repeats")
    ax.grid(axis="x", lw=0.6, color=GRID)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_yticks(ypos)
axes[0].set_yticklabels(exp_table.index)
axes[0].invert_yaxis()
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.9))
plt.show()

## 14. Predictor availability tiers

In [ ]:
TIERS = {
    "T0": ["Month", "Sex"],
    "T1": ["Month", "FatNo", "ColorNo", "Fatcol", "Texture", "Maturity", "Sex"],
    "T2": FEATURES,
}
fold_t0, oof_t0 = run_nested_cv(data, TIERS["T0"], MODEL_SPECS, REPEAT_SEEDS, "T0")
fold_t1, oof_t1 = run_nested_cv(data, TIERS["T1"], MODEL_SPECS, REPEAT_SEEDS, "T1")
fold_dummy, oof_dummy = run_nested_cv(data, FEATURES, [DUMMY_SPEC], REPEAT_SEEDS, "Dummy")

tier_tabs = {"Dummy": repeat_table(fold_dummy), "T0": repeat_table(fold_t0),
             "T1": repeat_table(fold_t1), "T2": repeat_table(fold_main)}
rows = []
for tier, tab in tier_tabs.items():
    rec = {"Tier": tier, "Best MAE model": tab["MAE"].idxmin(), "Best MAE": tab["MAE"].min(),
           "Best R2 model": tab["R2"].idxmax(), "Best R2": tab["R2"].max()}
    for m in ["Lasso", "SVR"]:
        if m in tab.index:
            rec[f"{m} MAE"], rec[f"{m} R2"] = tab.loc[m, "MAE"], tab.loc[m, "R2"]
    rows.append(rec)
tier_summary = pd.DataFrame(rows).set_index("Tier")
display(tier_summary)

rows = []
for m in ["Lasso", "SVR", "Ridge", "CatBoost"]:
    both = pd.concat([fold_main[fold_main["Model"] == m].assign(Model="T2"), fold_t1[fold_t1["Model"] == m].assign(Model="T1")])
    d_mae, _, p_mae = corrected_ttest(paired_fold_diff(both, "T2", "T1", "MAE"))
    d_r2, _, p_r2 = corrected_ttest(paired_fold_diff(both, "T2", "T1", "R2"))
    rows.append({"Model": m, "dMAE (T2 - T1)": d_mae, "p MAE": p_mae, "dR2 (T2 - T1)": d_r2, "p R2": p_r2})
display(pd.DataFrame(rows).set_index("Model"))
print(f"eta2 (backfat ~ Sex): {eta_squared(data[TARGET], data['Sex']):.3f}")

In [ ]:
def tolerance_table(oof_list):
    rows = []
    for label, o in oof_list:
        err = (o["y"] - o["pred"]).abs()
        rec = {"Model": label}
        for tol in [2, 3, 5]:
            rec[f"<= {tol} mm"] = (err <= tol).groupby(o["Repeat"]).mean().mean()
        rec["MAE"] = err.groupby(o["Repeat"]).mean().mean()
        rows.append(rec)
    return pd.DataFrame(rows).set_index("Model")


display(tolerance_table([
    ("Dummy", oof_dummy),
    ("Lasso T0", oof_t0[oof_t0["Model"] == "Lasso"]),
    ("Lasso T1", oof_t1[oof_t1["Model"] == "Lasso"]),
    ("Lasso T2", oof_main[oof_main["Model"] == "Lasso"]),
    ("SVR T2", oof_main[oof_main["Model"] == "SVR"]),
]))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.0))
tiers_plot = ["T0", "T1", "T2"]
x = np.arange(len(tiers_plot))
for ax, metric, ylabel in zip(axes, ["MAE", "R2"], ["MAE (mm)", "R²"]):
    for m, color, marker in [("Lasso", ORANGE, "o"), ("SVR", BLUE, "s")]:
        ax.plot(x, [tier_tabs[t].loc[m, metric] for t in tiers_plot], color=color, lw=2, marker=marker, ms=8,
                label=m, zorder=3, markeredgecolor="white", markeredgewidth=0.8)
    baseline = tier_tabs["Dummy"][metric]
    ax.axhline(baseline.min() if metric == "MAE" else baseline.max(), color=INK2, ls="--", lw=1.2, label="Dummy")
    ax.set_xticks(x)
    ax.set_xticklabels(["T0\nMonth, Sex", "T1\n+ grading scores", "T2\n+ agdays, WBsf"])
    ax.set_xlim(-0.4, len(x) - 0.6)
    ax.set_ylabel(f"{ylabel}, mean of {N_REPEATS} repeats")
    ax.grid(axis="y", lw=0.6, color=GRID)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.9))
plt.show()

In [ ]:
SUBSET = [SPEC_BY_NAME[m] for m in ["Lasso", "Ridge", "SVR", "RandomForest", "CatBoost"]]
fold_loin, _ = run_nested_cv(data, NUMERIC + ["Loinarea", "Sex"], SUBSET, REPEAT_SEEDS, "T2 + Loinarea")
fold_leak, _ = run_nested_cv(data, NUMERIC + ["qngrade", "Sex"], SUBSET, REPEAT_SEEDS, "T2 + qngrade")

names = [s.name for s in SUBSET]
display(pd.concat({
    "T2": repeat_table(fold_main).loc[names, ["MAE", "R2"]],
    "T2 + Loinarea": repeat_table(fold_loin).loc[names, ["MAE", "R2"]],
    "T2 + qngrade": repeat_table(fold_leak).loc[names, ["MAE", "R2"]],
}, axis=1))
print(f"r(backfat, Loinarea) = {stats.pearsonr(data['backfat'], data['Loinarea']).statistic:+.3f} | "
      f"r(backfat, qngrade) = {stats.pearsonr(data['backfat'], data['qngrade']).statistic:+.3f}")

## 15. Consistent-record sensitivity

In [ ]:
data_clean = data[data["record_conflict"] == 0]
fold_clean, oof_clean = run_nested_cv(data_clean, FEATURES, MODEL_SPECS, REPEAT_SEEDS, "Consistent records")
tab_clean = repeat_table(fold_clean)
cmp_clean = pd.concat({f"All ({len(data)})": tab_all[["MAE", "R2", "rank_MAE", "rank_R2"]],
                       f"Consistent ({len(data_clean)})": tab_clean[["MAE", "R2", "rank_MAE", "rank_R2"]]}, axis=1).loc[order]
cmp_clean[("Difference", "MAE")] = cmp_clean[(f"Consistent ({len(data_clean)})", "MAE")] - cmp_clean[(f"All ({len(data)})", "MAE")]
cmp_clean[("Difference", "R2")] = cmp_clean[(f"Consistent ({len(data_clean)})", "R2")] - cmp_clean[(f"All ({len(data)})", "R2")]
display(cmp_clean)
print(f"Spearman rho: MAE {stats.spearmanr(tab_all.loc[order, 'MAE'], tab_clean.loc[order, 'MAE']).statistic:.2f}, "
      f"R2 {stats.spearmanr(tab_all.loc[order, 'R2'], tab_clean.loc[order, 'R2']).statistic:.2f}")
display(seed_table(fold_clean, 0))

## 16. Summary

In [ ]:
lines = ["[Data]"]
lines += [f"  {r['Step']}: {r['Animals']} animals ({r['Animals excluded']} excluded)" for _, r in flow_df.iterrows()]
lines.append(f"  Conflicting animal-level records: {len(INCONSISTENT_IDS)}")
lines.append("[SVR vs Lasso]")
lines.append(f"  5-fold: paired t p = {t_res.pvalue:.3f}, Wilcoxon p = {w_res.pvalue:.3f}, min Wilcoxon p = {w_min:.4f}, "
             f"detectable difference = {d_detect * sd_diff:.3f} mm")
lines.append(f"  Bootstrap dMAE = {point['MAE (SVR - Lasso)']:+.3f} mm "
             f"[{boot_ci.loc['MAE (SVR - Lasso)', 'CI 2.5%']:+.3f}, {boot_ci.loc['MAE (SVR - Lasso)', 'CI 97.5%']:+.3f}]")
for metric in ["MAE", "R2"]:
    mean_d, _, p_c = corrected_ttest(paired_fold_diff(fold_main, "SVR", "Lasso", metric))
    lines.append(f"  {N_REPEATS}x5 CV: d{metric} = {mean_d:+.3f}, corrected p = {p_c:.3f}")
lines.append("[Ranking stability]")
lines.append(f"  Kendall's W: MAE {w_mae:.2f}, R2 {w_r2:.2f}; best MAE {stab['MAE'].idxmin()} ({stab['MAE'].min():.3f}), "
             f"best R2 {stab['R2'].idxmax()} ({stab['R2'].max():.3f})")
lines.append("[Subsets]")
for label, tab in [("No bulls", tab_nb), ("Steers only", tab_st), ("Consistent records", tab_clean)]:
    lines.append(f"  {label}: rho MAE {stats.spearmanr(tab_all.loc[order, 'MAE'], tab.loc[order, 'MAE']).statistic:.2f}, "
                 f"rho R2 {stats.spearmanr(tab_all.loc[order, 'R2'], tab.loc[order, 'R2']).statistic:.2f}; "
                 f"best MAE {tab['MAE'].idxmin()} ({tab['MAE'].min():.3f}), best R2 {tab['R2'].idxmax()} ({tab['R2'].max():.3f})")
lines.append("[Expanded search]")
for m, r in exp_table.iterrows():
    lines.append(f"  {m:17s} dMAE {r['dMAE']:+.3f}, dR2 {r['dR2']:+.3f}, R2 - Lasso {r['R2 - Lasso']:+.3f}")
lines.append("[Tiers]")
for t, r in tier_summary.iterrows():
    lines.append(f"  {t:5s} best R2 {r['Best R2']:.3f} ({r['Best R2 model']}), best MAE {r['Best MAE']:.3f} ({r['Best MAE model']})")
print("\n".join(lines))